# Mortality imputation leakage sensitivity analysis

This experiment tests whether outcome-informed imputation performed before the canonical 80/20 split made held-out mortality predictions look better than they would under a deployable imputation workflow. It follows the data, model registry, horizons, and validation conventions of `prediction23_converted_mod.ipynb`.

**Critical IBS warning.** The historical `ibs_ipcw_train()` used by `evaluate_dual_cox_python_style_boot.R` requests patient-specific censoring times from `summary.survfit()` without restoring their original row order. The holdout times are not sorted, so the saved final IBS values are affected. This notebook uses an order-safe lookup and includes a failing regression test. Existing thesis IBS values must be recalculated before they are retained. This finding does not by itself invalidate Uno's C or DCA, and the historical production scripts are not modified by this notebook.

Primary success criteria:

1. Reconstruct the pre-imputation missingness mask from raw data and verify exact positional alignment with the seed-2125 split.
2. Compare current mortality performance in rows with complete versus imputed model predictors as a diagnostic, not as proof of leakage.
3. Compare the original leaky workflow with the same original Cox model applied to a cleanly imputed holdout, and with a fully clean fixed-formula pipeline.
4. Quantify changes in absolute risk, Uno's C, Brier/IBS, calibration, net benefit, and mortality decision-analytic intervention equivalents per 1,000.
5. Use paired patient bootstrap intervals and a seed-matched joint future-outcome permutation diagnostic.


## Design and interpretation

Four fixed-formula prediction arms are used:

- **LL, original**: the reported pre-split, outcome-informed imputations and the reported held-out predictions.
- **LC, holdout imputation repair**: the original development Cox model is preserved, but validation predictors are re-imputed by an imputer fitted only in development and never given validation outcomes. LL−LC combines changes in the imputer training sample, predictor set, and PMM realization. It is not a pure estimate of test-outcome feedback.
- **CC, clean fixed-formula pipeline**: the imputer is fitted in development without outcomes, the frozen final Cox formula is refitted in clean development data, and the same holdout is imputed and evaluated without its outcomes. LL−CC estimates the total change after repairing the final pipeline while keeping formulas fixed.
- **Seed-matched split-leaky permutation arm**: an outcome-informed imputer is fitted only in development. For every permutation, validation predictors are imputed twice with the same PMM seed. The four post-discharge event/time endpoints are either correctly aligned or permuted jointly as a row block. Admission-origin times are reconstructed with each person's recorded admission-to-discharge offset plus the permuted post-discharge time. This preserves the original aligned arm exactly, retains small recorded rounding differences, and avoids impossible temporal combinations. The true outcomes are restored before evaluation. This paired contrast is the most specific diagnostic of joint future-outcome feedback, but feeding deliberately incorrect outcomes is not a deployable counterfactual, so it does not estimate the magnitude of leakage.

The complete-versus-imputed subgroup comparison is descriptive because missingness groups differ in case mix, event prevalence, and censoring. It cannot by itself identify leakage. The historical executable `missRanger` formula was `. ~ .`, despite comments suggesting outcome exclusion, so the leaky comparator follows the executable code. The clean arms exclude both event/time endpoints and the two admission-origin outcome times before imputation.


In [1]:
.t0 <- Sys.time()

SEED <- 2125L
RUN_MODE <- "confirmatory"  # confirmatory | smoke | quick

if (!RUN_MODE %in% c("quick", "smoke", "confirmatory")) {
  stop("RUN_MODE must be quick, smoke, or confirmatory.")
}

config <- list(
  run_mode = RUN_MODE,
  seed = SEED,
  permutation_seed = 92125L,
  n_imputations = if (RUN_MODE == "smoke") 1L else 5L,
  pmm_k = 15L,
  num_trees = if (RUN_MODE == "smoke") 20L else 200L,
  maxiter = if (RUN_MODE == "smoke") 2L else 10L,
  predict_iter = if (RUN_MODE == "smoke") 2L else 10L,
  num_threads = max(1L, parallel::detectCores(logical = TRUE) - 2L),
  verbose = 1L,
  eval_times = c(3, 6, 12, 36, 60),
  n_permutations = if (RUN_MODE == "smoke") 5L else 50L
)
BOOTSTRAP_B <- if (RUN_MODE == "smoke") 50L else 500L
PRIMARY_DCA <- c(`36` = 0.03, `60` = 0.05)

print(config)
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


$run_mode
[1] "confirmatory"

$seed
[1] 2125

$permutation_seed
[1] 92125

$n_imputations
[1] 5

$pmm_k
[1] 15

$num_trees
[1] 200

$maxiter
[1] 10

$predict_iter
[1] 10

$num_threads
[1] 30

$verbose
[1] 1

$eval_times
[1]  3  6 12 36 60

$n_permutations
[1] 50

Elapsed: 0.000 minutes


## Setup and reproducible inputs

The helper script defines functions only. Binary outputs are written under `data/20241015_out/leakage_sensitivity`, never under `cons/`. The most recent valid prediction23 validation list is selected automatically. The July 19 file overwritten with `save.image()` is rejected rather than loaded into the global environment.


In [3]:
project_root

[1] "G:/My Drive/Alvacast/SISTRAT 2023"

In [4]:
.t0 <- Sys.time()

project_root <- normalizePath(gsub("[/\\]cons$", "", getwd()), winslash = "/", mustWork = TRUE)
helper_path <- file.path(project_root, "cons/_alt_scripts/imputation_leakage_sensitivity.R")
stopifnot(file.exists(helper_path))
source(helper_path)
inputs <- leakage_load_inputs(project_root)
leakage_source_project_engines(project_root)
dir.create(inputs$output_dir, recursive = TRUE, showWarnings = FALSE)

cat("Source validation:", inputs$validation_path, "\n")
cat("Output directory:", inputs$output_dir, "\n")
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


Source validation: G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/pred23_holdout_validation_2026_07_17.rds 
Output directory: G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/leakage_sensitivity 
Elapsed: 0.416 minutes


In [5]:
.t0 <- Sys.time()

stopifnot(
  nrow(inputs$raw) == 88152L,
  length(inputs$train_idx) == 70521L,
  length(inputs$validation_idx) == 17631L,
  sum(inputs$missing_mask) == 6365L,
  identical(intersect(setdiff(names(inputs$raw), inputs$leakage_cols), inputs$endpoint_cols), character(0))
)

print(inputs$audit, row.names = FALSE)
print(inputs$model_exposure_audit, row.names = FALSE)
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


                          quantity value
        raw_rows_before_exclusions 88504
 excluded_administrative_artifacts   137
            excluded_other_outcome   215
                     analysis_rows 88152
                  development_rows 70521
                   validation_rows 17631
  validation_any_missing_predictor  6365
    validation_complete_predictors 11266
      model n_model_raw_predictors validation_rows
 best_perf1                     36           17631
 best_perf2                      8           17631
 rows_with_any_model_predictor_missing percent_with_any_model_predictor_missing
                                  6365                               36.1011854
                                   121                                0.6862912
 deaths_in_exposed_rows percent_of_all_deaths_in_exposed_rows missing_cells
                    302                            39.7368421          7302
                      6                             0.7894737           121
 percent_m

The exposure mask is model-specific. The Full PH mortality model uses many raw predictors and exposes substantially more validation rows to imputation than the final SHAP model. A single 36.1% figure must not be applied to both models.


In [23]:
show_table <- function(data, caption, digits = 1L) {
  htmltools::browsable(
    htmltools::div(
      style = "max-height: 520px; overflow-y: auto; overflow-x: auto;",
      htmltools::HTML(
        knitr::kable(
          data,
          format = "html",
          digits = digits,
          caption = caption,
          escape = TRUE
        )
      )
    )
  )
}


In [17]:
inputs$output_dir<- gsub("/leakage_sensitivity", "/leaksens", "G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/leakage_sensitivity")

In [19]:
#| label: model-specific-missing-variables
.t0 <- Sys.time()
# For each model, count missing values in its validation-set predictors.
missing_by_variable <- do.call(rbind, lapply(names(inputs$model_raw_predictors), function(model) {
  columns <- inputs$model_raw_predictors[[model]]
  # Keep only validation rows and the variables used by this model.
  validation_raw <- inputs$raw[inputs$validation_idx, columns, drop = FALSE]
  data.frame(
    model = model,
    variable = columns,
    missing_n = vapply(validation_raw, function(x) sum(is.na(x)), integer(1)),
    missing_percent = 100 * vapply(validation_raw, function(x) mean(is.na(x)), numeric(1)),
    row.names = NULL
  )
}))
# Keep only variables that actually have missing values.
missing_by_variable <- missing_by_variable[missing_by_variable$missing_n > 0, ]
# Sort by model and by descending number of missings.
missing_by_variable <- missing_by_variable[order(missing_by_variable$model, -missing_by_variable$missing_n), ]
# Print and save the result.
print(missing_by_variable, row.names = FALSE)
utils::write.csv(missing_by_variable, file.path(inputs$output_dir, "model_specific_missing_variables.csv"), row.names = FALSE)
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

      model                variable missing_n missing_percent
 best_perf1            any_violence      3556     20.16902048
 best_perf1          first_sub_used      1362      7.72502978
 best_perf1   tipo_de_vivienda_rec2      1270      7.20322160
 best_perf1 tenure_status_household       864      4.90045942
 best_perf1       prim_sub_freq_rec        94      0.53315183
 best_perf1      ed_attainment_corr        80      0.45374624
 best_perf1      marital_status_rec        31      0.17582667
 best_perf1             any_phys_dx        24      0.13612387
 best_perf1             eva_consumo         3      0.01701548
 best_perf1                 eva_fam         3      0.01701548
 best_perf1           eva_relinterp         3      0.01701548
 best_perf1           eva_ocupacion         3      0.01701548
 best_perf1                  eva_sm         3      0.01701548
 best_perf1              eva_fisica         3      0.01701548
 best_perf1         eva_transgnorma         3      0.01701548
 best_pe

## Quick diagnostic using the reported predictions

This section runs in all modes. It compares current held-out metrics by model-specific missingness exposure. A better C-index or DCA in the imputed subgroup is only a warning signal because the subgroups have different observed risks.


In [20]:
#| label: leakage-quick-diagnostic
.t0 <- Sys.time()
# Use the default probability thresholds for the leakage diagnostic.
thresholds <- leakage_default_thresholds()
# Run the quick leakage diagnostic and save its outputs to disk.
quick <- leakage_quick_diagnostic(inputs, thresholds = thresholds)
leakage_save_quick_outputs(inputs, quick)
# Print any warning and the elapsed time.
cat(quick$warning, "\n")
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

Complete-versus-imputed subgroup contrasts are diagnostic only. They mix leakage with case-mix and missingness-pattern differences. 
Elapsed: 0.115 minutes


In [25]:
#| label: brier-cross-engine-audit
.t0 <- Sys.time()
# Display selected metrics for key horizons.
quick_metrics_display <- subset(
  quick$metrics,
  horizon %in% c(6, 12, 36, 60),
  select = c(model, group, horizon, n, deaths_by_horizon, observed_risk,
             mean_predicted_risk, predicted_observed_ratio, uno_c, brier)
)
print(quick_metrics_display, row.names = FALSE)
# Extract Brier from the external validation object and clean model names.
external_brier <- subset(
  inputs$validation_object$ipeval$pooled,
  risk == "death",
  select = c(model, horizon, brier_mean)
)
external_brier$model <- sub("^death::", "", external_brier$model)
# Pull the corresponding Brier values from the quick diagnostic.
helper_brier <- subset(
  quick_metrics_display,
  group == "all",
  select = c(model, horizon, brier)
)
# Compare Brier values across the two engines and compute absolute differences.
brier_cross_engine_audit <- merge(
  helper_brier, external_brier,
  by = c("model", "horizon"), all = TRUE
)
brier_cross_engine_audit$absolute_difference <- abs(
  brier_cross_engine_audit$brier - brier_cross_engine_audit$brier_mean
)
knitr::kable(brier_cross_engine_audit, row.names = FALSE, "markdown", caption="Metrics for horizons")
# Sanity checks: complete comparisons, finite differences, and close agreement.
stopifnot(nrow(brier_cross_engine_audit) == 8L)
stopifnot(all(is.finite(brier_cross_engine_audit$absolute_difference)))
stopifnot(max(brier_cross_engine_audit$absolute_difference) < 0.002)
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

      model                       group horizon     n deaths_by_horizon
 best_perf1                         all       6 17631                99
 best_perf1                         all      12 17631               152
 best_perf1                         all      36 17631               418
 best_perf1                         all      60 17631               571
 best_perf1 any_model_predictor_imputed       6  6365                26
 best_perf1 any_model_predictor_imputed      12  6365                49
 best_perf1 any_model_predictor_imputed      36  6365               152
 best_perf1 any_model_predictor_imputed      60  6365               213
 best_perf1   complete_model_predictors       6 11266                73
 best_perf1   complete_model_predictors      12 11266               103
 best_perf1   complete_model_predictors      36 11266               266
 best_perf1   complete_model_predictors      60 11266               358
 best_perf2                         all       6 17631           

In [26]:
#| label: quick-dca-primary
.t0 <- Sys.time()
# Extract key DCA rows: 36-month at 3% threshold and 60-month at 5% threshold.
quick_dca_primary <- quick$dca[
  (quick$dca$horizon == 36 & abs(quick$dca$threshold - 0.03) < 1e-12) |
  (quick$dca$horizon == 60 & abs(quick$dca$threshold - 0.05) < 1e-12),
  c("model", "group", "horizon", "threshold", "positive_rate",
    "net_cases_per1000", "avoided_per1000_vs_all")
]
print(quick_dca_primary, row.names = FALSE)
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

      model                       group horizon threshold positive_rate
 best_perf1                         all      36      0.03     0.2735523
 best_perf1                         all      60      0.05     0.2674267
 best_perf1 any_model_predictor_imputed      36      0.03     0.2801257
 best_perf1 any_model_predictor_imputed      60      0.05     0.2743126
 best_perf1   complete_model_predictors      36      0.03     0.2698385
 best_perf1   complete_model_predictors      60      0.05     0.2635363
 best_perf2                         all      36      0.03     0.2834212
 best_perf2                         all      60      0.05     0.2772390
 best_perf2 any_model_predictor_imputed      36      0.03     0.2148760
 best_perf2 any_model_predictor_imputed      60      0.05     0.2231405
 best_perf2   complete_model_predictors      36      0.03     0.2838949
 best_perf2   complete_model_predictors      60      0.05     0.2776128
 net_cases_per1000 avoided_per1000_vs_all
         11.790803    

## Confirmatory counterfactual experiment

The notebook runs in `confirmatory` mode by default: five imputations with PMM k=15, 200 trees, 10 iterations, and 50 joint future-outcome permutations. Change `RUN_MODE` to `smoke` only for a structural test with one imputation, 20 trees, two iterations, and five permutations.

The confirmatory run can take several hours. It processes one imputation at a time and discards random forests, completed datasets, and fitted Cox objects after use. Only aggregate summaries and a session manifest are saved.


In [ ]:
#| label: counterfactual-run
.t0 <- Sys.time()
# Run the counterfactual sensitivity analysis unless in quick mode; reuse a cached copy when available.
counterfactual_cache <- file.path(inputs$output_dir, sprintf("counterfactual_%s.rds", RUN_MODE))
counterfactual <- NULL
if (RUN_MODE != "quick") {
  if (file.exists(counterfactual_cache)) {
    message("Loading cached counterfactual from ", counterfactual_cache)
    counterfactual <- readRDS(counterfactual_cache)
  } else {
    counterfactual <- leakage_counterfactual(inputs, config)
    saveRDS(counterfactual, counterfactual_cache, compress = "gzip")
  }
} else {
  message("Counterfactual run skipped in quick mode. Set RUN_MODE to smoke or confirmatory.")
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

Imputation 1/5: clean development-only imputer
Missing value imputation by random forests

Variables to impute:		sex_rec, evaluacindelprocesoteraputico, eva_consumo, eva_fam, eva_relinterp, eva_ocupacion, eva_sm, eva_fisica, eva_transgnorma, phys_dx_instudy, any_phys_dx, marital_status_rec, ed_attainment_corr, prim_sub_freq_rec, age_subs_onset_rec2, tenure_status_household, num_trat_ant, tipo_de_vivienda_rec2, first_sub_used, any_violence
Variables used to impute:	adm_age_rec3, porc_pobr, dit_m, num_trat_ant, age_subs_onset_rec2, sex_rec, tenure_status_household, cohabitation, sub_dep_icd10_status, any_violence, prim_sub_freq_rec, tr_outcome, adm_motive, first_sub_used, primary_sub_mod, tipo_de_vivienda_rec2, national_foreign, plan_type_corr, occupation_condition_corr24, marital_status_rec, urbanicity_cat, ed_attainment_corr, evaluacindelprocesoteraputico, eva_consumo, eva_fam, eva_relinterp, eva_ocupacion, eva_sm, eva_fisica, eva_transgnorma, ethnicity, dg_psiq_cie_10_instudy, dg_psiq

In [33]:
#| label: save-counterfactual-cache
.t0 <- Sys.time()
# Persist the expensive counterfactual object so a session crash does not force a re-run.
counterfactual_cache <- file.path(inputs$output_dir, sprintf("counterfactual_%s.rds", RUN_MODE))
saveRDS(counterfactual, counterfactual_cache, compress = "gzip")
cat("Saved:", counterfactual_cache, "\n")
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

Saved: G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/leaksens/counterfactual_confirmatory.rds 
Elapsed: 1.258 minutes


~ 1341 minutes

Derives scenario-level metrics, DCA, prediction shifts, threshold crossings, permutation tests, and optimism decomposition from the counterfactual object, or leaves everything NULL when it was not run.

Runs the paired bootstrap comparing leakage scenarios using the configured number of replicates, seed, and primary DCA thresholds, when the counterfactual object exists.

In [29]:
#| label: paired-bootstrap
.t0 <- Sys.time()
# Run the paired bootstrap over counterfactual scenarios when available.
paired_bootstrap <- NULL
if (!is.null(counterfactual)) {
  paired_bootstrap <- leakage_paired_bootstrap(
    counterfactual,
    b = BOOTSTRAP_B,
    seed = SEED,
    primary_thresholds = PRIMARY_DCA
  )
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

Elapsed: 37.142 minutes


## Predictive skill against null models

This section asks whether the covariates add held-out predictive information after correcting the imputation workflow. It evaluates three models separately: mortality Full PH, mortality SHAP rule2, and the shared readmission model. No skill estimate in this section uses the original pre-split imputation.

Two prespecified references answer different questions:

- `marginal_null` contains no covariates and estimates total gain over the development baseline hazard.
- `structure_null` retains only the original `strata()` terms and estimates incremental gain from the remaining covariates. A stratification variable can itself carry prognostic information, so this is not a zero-information reference.

The primary skill measures are the Brier skill score, `1 - BS_model / BS_null`, and its integrated version. Each ratio is accompanied by the absolute reduction `BS_null - BS_model`, because ratio skill can be unstable when the null error is small. Positive values favor the model. Delta Uno C and mortality decision-curve contrasts are supplementary. Readmission Brier and C statistics target only the cause-specific net-risk model, with competing death treated as censoring. They do not estimate predictive skill for the real-world cumulative incidence of readmission, and no readmission intervention-equivalent result is reported.

The structural-null event audit is printed and saved for every imputation. Mortality contrasts at 6 and 12 months are secondary because some structural strata contain few early deaths. The prespecified 36- and 60-month contrasts remain the main mortality comparisons.


In [30]:
#| label: model-skill-summaries
.t0 <- Sys.time()
# Model-skill definitions and point estimates; NULL when counterfactual is unavailable.
model_skill_definitions <- model_skill_point <- NULL
SKILL_THRESHOLDS <- leakage_model_skill_thresholds()
if (!is.null(counterfactual)) {
  # Build the model-skill definitions and the point estimates per model/threshold.
  model_skill_definitions <- leakage_model_skill_definitions(counterfactual)
  model_skill_point <- leakage_model_skill_point(
    counterfactual,
    thresholds = SKILL_THRESHOLDS
  )
  print(model_skill_definitions, row.names = FALSE)
  # Report the minimum number of events within each stratum (sample-size audit).
  minimum_events_by_stratum <- stats::aggregate(
    events_by_horizon ~ model_id + outcome + horizon,
    data = counterfactual$model_skill$strata_event_audit,
    FUN = min
  )
  print(minimum_events_by_stratum, row.names = FALSE)
  # Show Brier skill, delta metrics, and delta Uno's C at key horizons.
  print(subset(
    model_skill_point,
    metric %in% c("integrated_brier_skill_score", "delta_integrated_brier",
                  "brier_skill_score", "delta_brier", "delta_uno_c") &
      (is.infinite(horizon) | horizon %in% c(36, 60))
  ), row.names = FALSE)
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

             model_id                    label outcome registry_model
    mortality_full_ph        Mortality Full PH   death     best_perf1
 mortality_shap_rule2     Mortality SHAP rule2   death     best_perf2
          readmission Readmission shared model readmit     best_perf1
                            estimand               evaluation_scenario
            all-cause mortality risk clean_development_only_imputation
            all-cause mortality risk clean_development_only_imputation
 cause-specific net readmission risk clean_development_only_imputation
                                                                                                                                                                                                                                                                                                                                                                                                                                                    

Runs the bootstrap analysis for model-skill metrics and prints summaries for the integrated Brier skill score and decision-analytic interventions avoided per 1000.

In [31]:
#| label: model-skill-bootstrap
.t0 <- Sys.time()
# Bootstrap inference for model-skill metrics when the counterfactual exists.
model_skill_bootstrap <- NULL
if (!is.null(counterfactual)) {
  model_skill_bootstrap <- leakage_model_skill_bootstrap(
    counterfactual,
    b = BOOTSTRAP_B,
    seed = SEED + 40000L,
    thresholds = SKILL_THRESHOLDS
  )
  # Show key bootstrap summaries for the integrated Brier skill score and DCA interventions avoided.
  print(subset(
    model_skill_bootstrap,
    metric %in% c("integrated_brier_skill_score",
                  "decision_analytic_interventions_avoided_per1000_vs_null")
  ), row.names = FALSE)
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

             model_id                    label outcome registry_model
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
    mortality_full_ph        Mortality Full PH   death     best_perf1
 mortality_shap_rule2     Mortality SHAP rule2   death     best_perf2
 mortality_shap_rule2     Mortality SHAP rule2   death     best_perf2
 mortality_shap_rule2     Mortality SHAP rule2   death     best_perf2
 mortality_shap_rule

~ 133 minutes

### Audit checks for the skill module

The next cell deliberately stops on incomplete model coverage, invalid probabilities, nonconstant marginal-null predictions, non-stratification terms in a structural null, or disagreement between point estimates and the paired-bootstrap point calculations.


In [34]:
#| label: model-skill-audit

.t0 <- Sys.time()
# Audit the model-skill outputs for structure, dimension, and consistency.
if (!is.null(counterfactual)) {
  expected_models <- c("mortality_full_ph", "mortality_shap_rule2", "readmission")
  stopifnot(identical(names(counterfactual$model_skill$predictions), expected_models))
  stopifnot(nrow(model_skill_definitions) == 3L)
  stopifnot(nrow(model_skill_point) == 150L, nrow(model_skill_bootstrap) == 150L)
  # Each row must be uniquely keyed by model, null type, horizon, threshold, and metric.
  leakage_assert_unique_metric_keys(
    model_skill_point,
    c("model_id", "null_type", "horizon", "threshold", "metric"),
    "Point skill output"
  )
  leakage_assert_unique_metric_keys(
    model_skill_bootstrap,
    c("model_id", "null_type", "horizon", "threshold", "metric"),
    "Bootstrap skill output"
  )
  # Skill predictions must exactly match the corresponding clean predictions.
  stopifnot(isTRUE(all.equal(
    counterfactual$model_skill$predictions$mortality_full_ph$model,
    counterfactual$clean$best_perf1, tolerance = 1e-12
  )))
  stopifnot(isTRUE(all.equal(
    counterfactual$model_skill$predictions$mortality_shap_rule2$model,
    counterfactual$clean$best_perf2, tolerance = 1e-12
  )))
  stopifnot(all(c("marginal_null", "structure_null") %in%
                unique(model_skill_point$null_type)))
  # Every prediction matrix for a model must have the same dimensions and be valid probabilities.
  for (model_id in expected_models) {
    prediction_set <- counterfactual$model_skill$predictions[[model_id]]
    expected_dimension <- dim(prediction_set$model)
    for (prediction_name in names(prediction_set)) {
      stopifnot(identical(dim(prediction_set[[prediction_name]]), expected_dimension))
      stopifnot(all(is.finite(prediction_set[[prediction_name]])))
      stopifnot(all(prediction_set[[prediction_name]] >= 0 &
                    prediction_set[[prediction_name]] <= 1))
    }
    # Marginal-null predictions must be constant across rows.
    marginal_sd <- apply(prediction_set$marginal_null, 2L, stats::sd)
    stopifnot(max(marginal_sd, na.rm = TRUE) < 1e-10)
    # Structure-null formulas must only contain strata() terms.
    spec <- counterfactual$model_skill$specs[[model_id]]
    structure_terms <- attr(
      stats::terms(leakage_make_null_formula(spec$formula, "structure")),
      "term.labels"
    )
    stopifnot(!length(structure_terms) | all(grepl("^strata\\(", structure_terms)))
  }
  # Point estimates and bootstrap estimates must agree exactly for the same keys.
  consistency <- merge(
    model_skill_point[, c("model_id", "null_type", "horizon", "threshold",
                          "metric", "estimate")],
    model_skill_bootstrap[, c("model_id", "null_type", "horizon", "threshold",
                              "metric", "estimate")],
    by = c("model_id", "null_type", "horizon", "threshold", "metric"),
    suffixes = c("_point", "_bootstrap")
  )
  stopifnot(nrow(consistency) == nrow(model_skill_point))
  stopifnot(max(abs(consistency$estimate_point - consistency$estimate_bootstrap),
                na.rm = TRUE) < 1e-10)
  # At least 95% of requested bootstrap replicates must be valid.
  stopifnot(all(model_skill_bootstrap$b_valid >= 0.95 * model_skill_bootstrap$b_requested))
  cat("Model-skill audit checks passed.\n")
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

Model-skill audit checks passed.
Elapsed: 0.001 minutes


## Falsification and integrity checks

For LC, predictions among rows with complete model predictors should reproduce LL to numerical tolerance because the fitted Cox model is unchanged and no model input required imputation. A material difference in that negative-control group indicates a row-order, dummy-encoding, or model-reproduction error.


In [35]:
#| label: counterfactual-reproduction-audit
.t0 <- Sys.time()
# Audit the counterfactual prediction reproduction and the negative control identity.
if (!is.null(counterfactual)) {
  # Maximum absolute error when reproducing original predictions.
  reproduction_error <- subset(
    counterfactual$imputation_log,
    quantity == "max_abs_original_prediction_reproduction_error"
  )
  print(reproduction_error, row.names = FALSE)
  stopifnot(max(reproduction_error$value, na.rm = TRUE) < 1e-8)
  # For every model, compare the original leakage-free predictions with the clean validation predictions on complete rows.
  for (model in names(counterfactual$original)) {
    complete <- !counterfactual$model_missing_masks[[model]]
    negative_control_error <- max(abs(
      counterfactual$original[[model]][complete, , drop = FALSE] -
      counterfactual$original_model_clean_validation[[model]][complete, , drop = FALSE]
    ), na.rm = TRUE)
    cat(model, "complete-row max |LL-LC| =", format(negative_control_error, scientific = TRUE), "\n")
    stopifnot(negative_control_error < 1e-10)
  }
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

 imputation      model                                       quantity value
          1 best_perf1 max_abs_original_prediction_reproduction_error     0
          1 best_perf2 max_abs_original_prediction_reproduction_error     0
          2 best_perf1 max_abs_original_prediction_reproduction_error     0
          2 best_perf2 max_abs_original_prediction_reproduction_error     0
          3 best_perf1 max_abs_original_prediction_reproduction_error     0
          3 best_perf2 max_abs_original_prediction_reproduction_error     0
          4 best_perf1 max_abs_original_prediction_reproduction_error     0
          4 best_perf2 max_abs_original_prediction_reproduction_error     0
          5 best_perf1 max_abs_original_prediction_reproduction_error     0
          5 best_perf2 max_abs_original_prediction_reproduction_error     0
best_perf1 complete-row max |LL-LC| = 1.110223e-16 
best_perf2 complete-row max |LL-LC| = 1.110223e-16 
Elapsed: 0.000 minutes


Shows the decomposed optimism, paired bootstrap, and permutation-test results for the primary model (`best_perf1`) at the 36-month/3% and 60-month/5% decision points.

In [36]:
#| label: primary-leakage-components
.t0 <- Sys.time()
# Display the primary leakage component summaries for the best_perf1 model at 36/3% and 60/5%.
if (!is.null(counterfactual)) {
  primary_components <- component_optimism[
    component_optimism$model == "best_perf1" & (
      (component_optimism$horizon == 36 & (is.na(component_optimism$threshold) | abs(component_optimism$threshold - 0.03) < 1e-12)) |
      (component_optimism$horizon == 60 & (is.na(component_optimism$threshold) | abs(component_optimism$threshold - 0.05) < 1e-12))
    ),
  ]
  print(primary_components, row.names = FALSE)
  print(subset(paired_bootstrap, model == "best_perf1" & horizon %in% c(36, 60)), row.names = FALSE)
  print(subset(permutation_test, model == "best_perf1" & horizon %in% c(36, 60) &
               (is.na(threshold) | threshold %in% c(0.03, 0.05))), row.names = FALSE)
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

                            comparison      model horizon threshold
 holdout_imputation_repair_LL_minus_LC best_perf1      36        NA
 holdout_imputation_repair_LL_minus_LC best_perf1      60        NA
 holdout_imputation_repair_LL_minus_LC best_perf1      36        NA
 holdout_imputation_repair_LL_minus_LC best_perf1      60        NA
 holdout_imputation_repair_LL_minus_LC best_perf1      36      0.03
 holdout_imputation_repair_LL_minus_LC best_perf1      60      0.05
 holdout_imputation_repair_LL_minus_LC best_perf1      36      0.03
 holdout_imputation_repair_LL_minus_LC best_perf1      60      0.05
     total_pipeline_change_LL_minus_CC best_perf1      36        NA
     total_pipeline_change_LL_minus_CC best_perf1      60        NA
     total_pipeline_change_LL_minus_CC best_perf1      36        NA
     total_pipeline_change_LL_minus_CC best_perf1      60        NA
     total_pipeline_change_LL_minus_CC best_perf1      36      0.03
     total_pipeline_change_LL_minus_CC best_perf

Displays the imputed-value log, prediction shifts, and threshold-crossing counts for `best_perf1` at the 36-month/3% and 60-month/5% thresholds.

In [37]:
#| label: leakage-imputation-and-threshold-crossings
.t0 <- Sys.time()
# Show imputation details, prediction shifts, and threshold crossings for best_perf1 at primary horizons.
if (!is.null(counterfactual)) {
  print(counterfactual$imputed_value_log, row.names = FALSE)
  print(prediction_shifts, row.names = FALSE)
  print(subset(threshold_crossings, model == "best_perf1" & group == "all" &
               ((horizon == 36 & abs(threshold - 0.03) < 1e-12) |
                (horizon == 60 & abs(threshold - 0.05) < 1e-12))), row.names = FALSE)
}
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

 imputation                variable        type n_originally_missing agreement
          1 tenure_status_household categorical                  864 0.3182870
          1            any_violence categorical                 3556 0.5961755
          1       prim_sub_freq_rec categorical                   94 0.4787234
          1          first_sub_used categorical                 1362 0.4456681
          1   tipo_de_vivienda_rec2 categorical                 1270 0.7055118
          1      marital_status_rec categorical                   31 0.7096774
          1      ed_attainment_corr categorical                   80 0.3500000
          1             eva_consumo categorical                    3 0.6666667
          1                 eva_fam categorical                    3 0.6666667
          1           eva_relinterp categorical                    3 0.6666667
          1           eva_ocupacion categorical                    3 0.6666667
          1                  eva_sm categorical     

## Reading the DCA contrasts

Positive LL−LC or LL−CC net-benefit differences indicate that the original workflow appeared more useful. For unnecessary intensive interventions avoided:

`avoided/1,000 = (NB_model − NB_all) / (pt / (1 − pt)) × 1,000`.

At low mortality thresholds this transformation amplifies small net-benefit differences. For example, at a 1% threshold, a net-benefit difference of 0.001 corresponds to approximately 99 interventions per 1,000. Therefore, always report the change in net benefit, net cases per 1,000, proportion selected, and avoided interventions together. These are decision-analytic equivalents, not interventions observed in practice, and they must not be used to restrict ordinary care.


## Decision rules

Evidence compatible with meaningful leakage is present when several findings agree:

- LL outperforms LC and CC, with paired intervals excluding changes that are negligible for the intended use.
- Prediction changes are concentrated in rows whose model predictors were originally missing, while the complete-row negative control remains numerically unchanged in LL versus LC.
- Correctly aligned future outcomes outperform jointly permuted readmission and mortality outcomes under the same outcome-informed imputer and matched PMM seed.
- Net benefit or interventions avoided materially decline at the frozen 36-month/3% and 60-month/5% operating points.

Absence of a complete-versus-imputed subgroup difference does not prove absence of leakage. A paired interval containing zero also does not establish equivalence. Report the sign, magnitude, and uncertainty without forcing the result to be optimistic.


In [38]:
.t0 <- Sys.time()

if (!is.null(counterfactual)) {
  leakage_save_counterfactual_outputs(
    inputs = inputs,
    counterfactual = counterfactual,
    metrics = scenario_metrics,
    dca = scenario_dca,
    shifts = prediction_shifts,
    permutation = permutation_test,
    optimism = total_optimism,
    component_optimism = component_optimism,
    paired_bootstrap = paired_bootstrap,
    threshold_crossings = threshold_crossings
  )
  leakage_save_model_skill_outputs(
    inputs = inputs,
    counterfactual = counterfactual,
    point_estimates = model_skill_point,
    paired_bootstrap = model_skill_bootstrap
  )
}
cat("Outputs available at:", inputs$output_dir, "\n")
print(utils::sessionInfo())
cat(sprintf("Elapsed: %.3f minutes\n", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))


Outputs available at: G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/leaksens 
R version 4.4.1 (2024-06-14 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26200)

Matrix products: default


locale:
[1] LC_COLLATE=Spanish_Chile.utf8  LC_CTYPE=Spanish_Chile.utf8   
[3] LC_MONETARY=Spanish_Chile.utf8 LC_NUMERIC=C                  
[5] LC_TIME=Spanish_Chile.utf8    

time zone: America/Santiago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] tidyselect_1.2.1          nanoparquet_0.4.2        
 [3] dplyr_1.1.4               farver_2.1.2             
 [5] S7_0.2.1                  fastmap_1.2.0            
 [7] TH.data_1.1-3             janitor_2.2.1            
 [9] digest_0.6.37             rpart_4.1.23             
[11] timechange_0.3.0          lifecycle_1.0.4          
[13] cluster_2.1.8.1           survival_3.6-4           
[15] ma

## Scope of inference

The paired bootstrap is conditional on the fixed split, frozen formulas, and generated imputations. It does not include uncertainty from repeating model selection, hyperparameter tuning, or the split. The 50 seed-matched permutations form a diagnostic Monte Carlo falsification. Their paired-gain quantiles describe variability across PMM seeds and outcome permutations, not confidence intervals, and the fraction of pairs not favoring aligned outcomes is not a calibrated p-value. If the fixed-formula correction changes conclusions materially, the full repair is to fit imputation inside every development resampling fold and leave the holdout untouched until the final evaluation.
